# VoltSight - Phase 5.2 : détecter les équipements avec YOLO

Objectif : trouver automatiquement les 17 types d'équipements (isolateurs, amortisseurs, manilles…) sur de vraies photos de drone InsPLAD.

Référence : l'article InsPLAD obtient une *Box AP* (mAP@0.5:0.95) de 0,721 avec DetectoRS, un modèle lourd. J'utilise YOLO11s, beaucoup plus léger et rapide, utilisable sur le terrain.


In [ ]:
# Cellule 1 : installation et vérification du GPU
!pip install -q ultralytics
import torch, os
print("GPU disponible :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun (activer un GPU T4)")
BASE = "/content"          # dossier de travail

## Données
Mendeley bloque le téléchargement direct depuis Colab. Si InsPLAD n'est pas déjà présent, la cellule demande le lien temporaire de téléchargement (donc je me ren sur Mendeley, puis copie de l'adresse du lien dans le navigateur).

In [ ]:
# Cellule 2 : télécharger InsPLAD (sautée si déjà présent dans cette session)
import os, glob, subprocess, zipfile

def trouver_det():
    r = glob.glob(f"{BASE}/insplad/**/InsPLAD-det", recursive=True)
    return r[0] if r else None

if trouver_det() or glob.glob(f"{BASE}/insplad/*"):
    print("Données déjà présentes dans cette session : pas de nouveau téléchargement.")
else:
    print("Lien temporaire Mendeley requis (Download All, puis copier l'adresse du lien).")
    LIEN = input("Lien de téléchargement : ").strip()
    cible = f"{BASE}/insplad_complet.zip"
    subprocess.run(["wget", "-q", "--show-progress", "-O", cible, LIEN])
    taille = os.path.getsize(cible) / 1e9
    print(f"Taille : {taille:.2f} Go")
    assert taille > 1, "Erreur : le lien a expiré, il faut en générer un nouveau."
    with zipfile.ZipFile(cible) as z:
        z.extractall(f"{BASE}/insplad")
    os.remove(cible)

# Les zips sont imbriqués (archive Mendeley -> InsPLAD_Dataset.zip -> InsPLAD-det.zip).
# On décompresse en boucle tant qu'il en reste, en ignorant les 2 parties « fault » (inutiles ici).
IGNORER = ("unsupervised_anomaly_detection.zip", "supervised_fault_classification.zip")
for _ in range(4):
    if trouver_det():
        break
    for racine, _, fs in os.walk(f"{BASE}/insplad"):
        for f in fs:
            if f.lower().endswith(".zip") and f not in IGNORER:
                z = os.path.join(racine, f)
                print("Décompression de", f, "...")
                zipfile.ZipFile(z).extractall(os.path.splitext(z)[0]); os.remove(z)
assert trouver_det(), "Erreur : dossier InsPLAD-det introuvable après extraction."
print("Prêt :", trouver_det())

In [ ]:
# Cellule 3 : lire les annotations COCO et retrouver les images
import json, os, glob, collections
DET = trouver_det()
ann_files = {"train": glob.glob(f"{DET}/**/instances_train.json", recursive=True)[0],
             "val":   glob.glob(f"{DET}/**/instances_val.json",   recursive=True)[0]}
coco = {s: json.load(open(p)) for s, p in ann_files.items()}

# index de toutes les images par nom de fichier
index = {}
for p in glob.glob(f"{DET}/**/*", recursive=True):
    if p.lower().endswith((".jpg", ".jpeg", ".png")):
        index.setdefault(os.path.basename(p), p)
print("Images trouvées sur le disque :", len(index))

# classes : uniquement celles qui ont des boîtes (on ignore une éventuelle « super-catégorie » vide)
cats = {c["id"]: c["name"] for c in coco["train"]["categories"]}
utilisees = sorted({a["category_id"] for s in coco for a in coco[s]["annotations"]})
CLASSES = [cats[i] for i in utilisees]
cat2yolo = {cid: k for k, cid in enumerate(utilisees)}
print(f"{len(CLASSES)} classes :", CLASSES)

for s in coco:
    manquantes = [im["file_name"] for im in coco[s]["images"] if os.path.basename(im["file_name"]) not in index]
    compte = collections.Counter(cats[a["category_id"]] for a in coco[s]["annotations"])
    print(f"\n{s} : {len(coco[s]['images'])} images ({len(manquantes)} introuvables), {len(coco[s]['annotations'])} boîtes")
    for nom, n in compte.most_common():
        print(f"   {n:6d}  {nom}")

## Conversion au format YOLO
Les annotations sont au format COCO (`[x, y, largeur, hauteur]` en pixels). YOLO veut un fichier `.txt` par image : `classe x_centre y_centre largeur hauteur`, en proportions de 0 à 1. Durée : environ 5 à 10 minutes.

In [ ]:
# Cellule 4 : convertir au format YOLO (images réduites + fichiers .txt)
# Les coordonnées YOLO sont relatives (entre 0 et 1) : réduire l'image ne change pas les étiquettes.
# Réduire les photos 1920 x 1080 une seule fois accélère énormément l'entraînement sur Colab.
import yaml
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

TAILLE_MAX = 1024     # côté le plus long après réduction (les petits objets restent visibles)
YOLO_DIR = f"{BASE}/yolo_insplad"

def convertir(split):
    os.makedirs(f"{YOLO_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_DIR}/labels/{split}", exist_ok=True)
    par_image = collections.defaultdict(list)
    for a in coco[split]["annotations"]:
        if not a.get("iscrowd", 0):
            par_image[a["image_id"]].append(a)
    taches = []
    for im in coco[split]["images"]:
        nom = os.path.basename(im["file_name"])
        if nom in index:
            taches.append((im, nom, par_image.get(im["id"], [])))

    def traiter(t):
        im, nom, anns = t
        src = index[nom]
        dst = f"{YOLO_DIR}/images/{split}/{nom}"
        with Image.open(src) as img:
            W, H = img.size
            if not os.path.exists(dst):
                img = img.convert("RGB")
                img.thumbnail((TAILLE_MAX, TAILLE_MAX))
                img.save(dst, quality=92)
        lignes = []
        for a in anns:
            x, y, w, h = a["bbox"]
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            lignes.append(f"{cat2yolo[a['category_id']]} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} "
                          f"{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
        with open(f"{YOLO_DIR}/labels/{split}/{os.path.splitext(nom)[0]}.txt", "w") as f:
            f.write("\n".join(lignes))
        return len(lignes)

    with ThreadPoolExecutor(8) as ex:
        n = sum(ex.map(traiter, taches))
    print(f"{split} : {len(taches)} images, {n} boîtes converties")

for s in ["train", "val"]:
    convertir(s)

with open(f"{YOLO_DIR}/data.yaml", "w") as f:
    yaml.safe_dump({"path": YOLO_DIR, "train": "images/train", "val": "images/val",
                    "names": {i: n for i, n in enumerate(CLASSES)}}, f, allow_unicode=True)
print(open(f"{YOLO_DIR}/data.yaml").read())

## Vérification
Les boîtes vertes doivent entourer exactement les équipements. Si elles sont décalées, la conversion des annotations est à revoir.

In [ ]:
# Cellule 5 : VÉRIFIER la conversion (toujours regarder ses données avant d'entraîner !)
import random, matplotlib.pyplot as plt, matplotlib.patches as patches
random.seed(1)
imgs = sorted(glob.glob(f"{YOLO_DIR}/images/train/*"))
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, p in zip(axs, random.sample(imgs, 3)):
    img = Image.open(p); W, H = img.size
    ax.imshow(img); ax.axis("off")
    lab = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    for ligne in open(lab).read().splitlines():
        c, xc, yc, w, h = ligne.split(); xc, yc, w, h = map(float, (xc, yc, w, h))
        ax.add_patch(patches.Rectangle(((xc - w / 2) * W, (yc - h / 2) * H), w * W, h * H,
                                       fill=False, ec="lime", lw=1.5))
        ax.text((xc - w / 2) * W, (yc - h / 2) * H - 4, CLASSES[int(c)], color="lime", fontsize=7)
plt.tight_layout(); plt.show()

# répartition des classes (le déséquilibre explique souvent les mauvaises classes)
compte = collections.Counter()
for lab in glob.glob(f"{YOLO_DIR}/labels/train/*.txt"):
    for ligne in open(lab).read().splitlines():
        compte[CLASSES[int(ligne.split()[0])]] += 1
noms, n = zip(*compte.most_common())
plt.figure(figsize=(10, 4)); plt.barh(noms[::-1], n[::-1], color="#1f77b4")
plt.xlabel("nombre de boîtes (train)"); plt.title("Répartition des 17 équipements"); plt.tight_layout(); plt.show()

## Entraînement
Apprentissage par transfert : YOLO11s sait déjà reconnaître des formes générales (pré-entraînement sur le jeu COCO). On l'adapte à nos 17 équipements. L'arrêt précoce (`patience=10`) stoppe l'entraînement si le score de validation ne progresse plus.

Sécurité : les poids sont enregistrés dans Google Drive à chaque époque. En cas de déconnexion, il suffit de relancer les cellules 1 à 4 puis la 6 : l'entraînement reprend où il s'était arrêté.

In [ ]:
# Cellule 6 : entraîner YOLO, avec sauvegarde AUTOMATIQUE dans Google Drive à chaque époque
# Si Colab se déconnecte, relancer les cellules 1 à 4 puis celle-ci : l'entraînement REPREND là où il s'était arrêté.
from ultralytics import YOLO
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJET = "/content/drive/MyDrive/VoltSight/runs"      # environ 100 Mo dans Google Drive
except ImportError:
    PROJET = f"{BASE}/runs"
os.makedirs(PROJET, exist_ok=True)

EPOQUES = 30          # le score plafonne vers 30 époques (essai du 6 octobre : 0,728 à l'époque 30)
IMGSZ = 640
NOM = "insplad_yolo11s"
dernier = f"{PROJET}/{NOM}/weights/last.pt"

if os.path.exists(dernier):
    print("Entraînement interrompu trouvé dans le Drive : reprise à partir de", dernier)
    model = YOLO(dernier)
    resultats = model.train(resume=True)
else:
    model = YOLO("yolo11s.pt")      # pré-entraîné sur COCO (apprentissage par transfert)
    resultats = model.train(data=f"{YOLO_DIR}/data.yaml", epochs=EPOQUES, imgsz=IMGSZ, batch=16,
                            patience=10, workers=2, seed=0, project=PROJET, name=NOM,
                            exist_ok=True, plots=True)
MEILLEUR = f"{PROJET}/{NOM}/weights/best.pt"
print("Meilleur modèle (sauvegardé dans Google Drive) :", MEILLEUR)

## Évaluation
Note d'honnêteté : comme dans l'article, on évalue sur leur ensemble `val`, qui sert ici d'ensemble de test. On n'a pas réglé d'hyperparamètres dessus.

In [ ]:
# Cellule 7 : évaluer sur l'ensemble de test officiel (instances_val.json, 2 626 images)
import pandas as pd
model = YOLO(MEILLEUR)
m = model.val(data=f"{YOLO_DIR}/data.yaml", imgsz=IMGSZ, split="val", plots=True,
              project=PROJET, name="insplad_eval", exist_ok=True)
print(f"\nmAP@0.5       : {m.box.map50:.3f}")
print(f"mAP@0.5:0.95  : {m.box.map:.3f}   <- même métrique que la « Box AP » de l'article")
print("Référence de l'article (DetectoRS) : 0.721\n")
par_classe = pd.DataFrame({"classe": [CLASSES[i] for i in m.box.ap_class_index],
                           "AP50": m.box.ap50, "AP50-95": m.box.ap}).sort_values("AP50-95")
print(par_classe.round(3).to_string(index=False))
par_classe.to_csv(f"{PROJET}/insplad_eval/ap_par_classe.csv", index=False)

In [ ]:
# Cellule 8 : voir les détections sur des photos de test
from IPython.display import Image as IImage, display
for f in ["confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
    p = f"{PROJET}/insplad_eval/{f}"
    if os.path.exists(p):
        print(f); display(IImage(p, width=800))
exemples = random.sample(sorted(glob.glob(f"{YOLO_DIR}/images/val/*")), 4)
for r in model.predict(exemples, imgsz=IMGSZ, conf=0.25, verbose=False):
    display(Image.fromarray(r.plot()[:, :, ::-1]).resize((960, 540)))

## Sauvegarde
 dans Google Drive (`VoltSight/runs`).

In [ ]:
# Cellule 9 : TÉLÉCHARGER les résultats
import shutil
sortie = f"{BASE}/resultats_yolo"
os.makedirs(sortie, exist_ok=True)
shutil.copy(MEILLEUR, f"{sortie}/insplad_yolo11s_best.pt")
for d in ["insplad_yolo11s", "insplad_eval"]:
    for f in glob.glob(f"{PROJET}/{d}/*.png") + glob.glob(f"{PROJET}/{d}/*.csv"):
        shutil.copy(f, f"{sortie}/{d}__{os.path.basename(f)}")
shutil.make_archive(f"{BASE}/resultats_yolo", "zip", sortie)
print("Taille :", round(os.path.getsize(f"{BASE}/resultats_yolo.zip") / 1e6, 1), "Mo")
from google.colab import files
files.download(f"{BASE}/resultats_yolo.zip")